In [46]:

from dotenv import load_dotenv
import os
load_dotenv()
from openai import OpenAI
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
client = OpenAI()

import json
from pathlib import Path
from openai import OpenAI

client = OpenAI()

# Pfade
calls_dir = Path("../../emergency_calls/calls_german").resolve()
guidelines_dir = Path("guidelines").resolve()

assert calls_dir.exists(), f"calls_dir nicht gefunden: {calls_dir}"
assert guidelines_dir.exists(), f"guidelines_dir nicht gefunden: {guidelines_dir}"

# Notrufe sammeln
call_files = sorted(list(calls_dir.glob("*.md")) + list(calls_dir.glob("*.txt")))
assert call_files, f"Keine Notrufe gefunden in {calls_dir}"

# Guidelines je Domäne laden
DOMAIN_FILES = {
    "A": guidelines_dir / "a_guidelines.md",
    "B": guidelines_dir / "b_guidelines.md",
    "C": guidelines_dir / "c_guidelines.md",
    "D": guidelines_dir / "d_guidelines.md",
    "E": guidelines_dir / "e_guidelines.md",
}
domain_guidelines = {}
for d, p in DOMAIN_FILES.items():
    assert p.exists(), f"Guideline fehlt: {p}"
    domain_guidelines[d] = p.read_text(encoding="utf-8").strip()

# Output Contract (minimal, pro Domain)
output_contract = """
Return ONLY valid JSON (no markdown, no commentary).

Schema:
{
  "severity": <int or null>,
  "confidence": <float 0..1 or null>,
  "findings": <list of strings>
}

Rules:
- findings must be short factual phrases grounded in the call text.
- if no findings are available, return [].
- do NOT invent findings.
"""

MODEL = "gpt-5.2"
TEMPERATURE = 1

batch_input_path = Path("batch_input_abcde.jsonl").resolve()

with batch_input_path.open("w", encoding="utf-8") as f:

    for fp in call_files:

        transcript = fp.read_text(encoding="utf-8").strip()

        for domain in ["A", "B", "C", "D", "E"]:
            req = {
            "custom_id": f"{fp.stem}__{domain}",
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": MODEL,
                "temperature": TEMPERATURE,
                "input": [
                    {"role": "system", "content": [{"type": "input_text", "text": domain_guidelines[domain]}]},
                    {"role": "system", "content": [{"type": "input_text", "text": output_contract}]},
                    {"role": "user",   "content": [{"type": "input_text", "text": transcript}]},
                ],
                # JSON erzwingen (wie zuvor)
                "text": {"format": {"type": "json_object"}},
            },
        }
            f.write(json.dumps(req, ensure_ascii=False) + "\n")


print("Geschrieben:", batch_input_path)
print("Notrufe:", len(call_files), "=> Requests:", len(call_files) * 5)

Geschrieben: /Users/s.franke/Development/master_clean/experiments/experiment_3/batch_input_abcde.jsonl
Notrufe: 60 => Requests: 300


In [47]:
upload = client.files.create(file=batch_input_path.open("rb"), purpose="batch")

batch = client.batches.create(
    input_file_id=upload.id,
    endpoint="/v1/responses",
    completion_window="24h"
)

print("batch_id:", batch.id, "status:", batch.status)

batch_id: batch_69a0cad4807481908986e3f28f7ca132 status: validating


In [52]:

b = client.batches.retrieve(batch.id)
print("status:", b.status)
print("output_file_id:", b.output_file_id)
print("error_file_id:", b.error_file_id)

status: completed
output_file_id: file-L4kyWCwLYWy14xeMDi377U
error_file_id: None


In [53]:
# Output herunterladen (wenn status == "completed")
out_path = Path("batch_output.jsonl").resolve()

b = client.batches.retrieve(batch.id)
assert b.output_file_id, "Noch kein output_file_id vorhanden (Batch evtl. noch nicht fertig)."

content = client.files.content(b.output_file_id)
out_path.write_bytes(content.read())

print("Gespeichert:", out_path)

Gespeichert: /Users/s.franke/Development/master_clean/experiments/experiment_3/batch_output.jsonl


In [54]:
import json
import re
import pandas as pd
from pathlib import Path

def _extract_json_from_response_obj(resp_body: dict) -> dict:
    out = resp_body.get("output", [])
    texts = []
    for item in out:
        for c in item.get("content", []):
            if c.get("type") in ("output_text", "text") and "text" in c:
                texts.append(c["text"])
    if not texts and resp_body.get("output_text"):
        texts = [resp_body["output_text"]]
    if not texts:
        raise ValueError("Kein output_text gefunden")

    text = "\n".join(texts).strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        raise ValueError(f"Kein JSON im Text: {text[:200]}...")
    return json.loads(m.group(0))

def parse_batch_output_to_df(batch_output_jsonl: str) -> pd.DataFrame:
    # Sammelstruktur: {Notruf_12: {"severity_A":..., ...}}
    by_call = {}

    lines = Path(batch_output_jsonl).read_text(encoding="utf-8").splitlines()
    for line in lines:
        if not line.strip():
            continue
        rec = json.loads(line)

        custom_id = rec.get("custom_id")  # Notruf_12__A
        if not custom_id or "__" not in custom_id:
            continue

        call_id, domain = custom_id.split("__", 1)
        domain = domain.upper()

        by_call.setdefault(call_id, {"file": call_id})

        err = rec.get("error")
        if err:
            by_call[call_id][f"severity_{domain}"] = None
            by_call[call_id][f"confidence_{domain}"] = None
            by_call[call_id][f"findings_{domain}"] = json.dumps([], ensure_ascii=False)
            by_call[call_id][f"error_{domain}"] = json.dumps(err, ensure_ascii=False)
            continue

        body = (rec.get("response") or {}).get("body") or {}
        try:
            data = _extract_json_from_response_obj(body)
            sev = data.get("severity")
            conf = data.get("confidence")
            findings = data.get("findings", [])
            if findings is None:
                findings = []
            if not isinstance(findings, list):
                findings = [str(findings)]
        except Exception as e:
            by_call[call_id][f"severity_{domain}"] = None
            by_call[call_id][f"confidence_{domain}"] = None
            by_call[call_id][f"findings_{domain}"] = json.dumps([], ensure_ascii=False)
            by_call[call_id][f"error_{domain}"] = f"parse_error: {e}"
            continue

        by_call[call_id][f"severity_{domain}"] = sev
        by_call[call_id][f"confidence_{domain}"] = conf
        by_call[call_id][f"findings_{domain}"] = json.dumps(findings, ensure_ascii=False)
        by_call[call_id][f"error_{domain}"] = None

    df = pd.DataFrame(list(by_call.values()))

    # call_nr für Sortierung
    df["call_nr"] = df["file"].str.extract(r"(\d+)").astype("Int64")
    df = df.sort_values(["call_nr", "file"], na_position="last").reset_index(drop=True)

    # Spaltenreihenfolge hübsch
    cols = ["file", "call_nr"]
    for d in ["A","B","C","D","E"]:
        cols += [f"severity_{d}", f"confidence_{d}", f"findings_{d}", f"error_{d}"]
    for c in cols:
        if c not in df.columns:
            df[c] = None
    df = df[cols]

    return df

df = parse_batch_output_to_df("batch_output.jsonl")
df.head()

,file,call_nr,severity_A,confidence_A,findings_A,error_A,severity_B,confidence_B,findings_B,error_B,...,findings_C,error_C,severity_D,confidence_D,findings_D,error_D,severity_E,confidence_E,findings_E,error_E
0,Notruf_1,1,1,0.72,"[""Caller speaks in full sentences throughout t...",None,2,0.78,"[""Caller reports 'kriege auch schlecht Luft' (...",None,...,"[""acute chest pain started a few minutes ago"",...",None,1,0.60,"[""caller speaks coherently and answers questio...",None,1,0.72,"[""Brustschmerzen seit ein paar Minuten"", ""schl...",None
1,Notruf_2,2,1,0.62,"[""normal ansprechbar"", ""reagiert ganz normal b...",None,1,0.93,"[""Caller: \""Nee, er atmet gut ja ganz normal\""...",None,...,"[""64-year-old with diabetes and high blood pre...",None,2,0.90,"[""Seit etwa einer Stunde verwaschene Sprache"",...",None,1,0.63,"[""keine Verletzung oder Unfallmechanismus gena...",None
2,Notruf_3,3,1,0.62,"[""Reagiert auf Ansprache"", ""Schläft ansonsten""...",None,1,0.62,"[""Patient reacts when spoken to"", ""Caller init...",None,...,"[""Mann liegt auf der Straße"", ""reagiert beim A...",None,2,0.72,"[""man lying on the street"", ""reacts when spoke...",None,1,0.62,"[""Mann liegt auf der Straße vor Primark (Bahnh...",None
3,Notruf_4,4,1,0.78,"[""caller speaks/responds throughout the call"",...",None,2,0.72,"[""Caller reports 'krieg schlecht Luft' (diffic...",None,...,"[""Caller reports no pain"", ""No dizziness/circu...",None,1,0.55,"[""caller answers questions and gives location""...",None,1,0.70,"[""kein Trauma/Unfallmechanismus genannt"", ""Auf...",None
4,Notruf_5,5,1,0.66,"[""Patientin ist normal bei Bewusstsein"", ""Kein...",None,1,0.62,"[""caller reports she sometimes pants a little ...",None,...,"[""normal bei Bewusstsein"", ""keine weiteren Sch...",None,1,0.78,"[""Patientin laut Disponent \""normal bei Bewuss...",None,3,0.64,"[""Schwangere in 34. Schwangerschaftswoche"", ""G...",None


In [55]:
out_csv = Path("run_3.csv").resolve()
df.to_csv(out_csv, index=False, encoding="utf-8")
print("Gespeichert:", out_csv)

Gespeichert: /Users/s.franke/Development/master_clean/experiments/experiment_3/run_3.csv


In [56]:
df

,file,call_nr,severity_A,confidence_A,findings_A,error_A,severity_B,confidence_B,findings_B,error_B,...,findings_C,error_C,severity_D,confidence_D,findings_D,error_D,severity_E,confidence_E,findings_E,error_E
0,Notruf_1,1,1,0.72,"[""Caller speaks in full sentences throughout t...",None,2,0.78,"[""Caller reports 'kriege auch schlecht Luft' (...",None,...,"[""acute chest pain started a few minutes ago"",...",None,1,0.60,"[""caller speaks coherently and answers questio...",None,1,0.72,"[""Brustschmerzen seit ein paar Minuten"", ""schl...",None
1,Notruf_2,2,1,0.62,"[""normal ansprechbar"", ""reagiert ganz normal b...",None,1,0.93,"[""Caller: \""Nee, er atmet gut ja ganz normal\""...",None,...,"[""64-year-old with diabetes and high blood pre...",None,2,0.90,"[""Seit etwa einer Stunde verwaschene Sprache"",...",None,1,0.63,"[""keine Verletzung oder Unfallmechanismus gena...",None
2,Notruf_3,3,1,0.62,"[""Reagiert auf Ansprache"", ""Schläft ansonsten""...",None,1,0.62,"[""Patient reacts when spoken to"", ""Caller init...",None,...,"[""Mann liegt auf der Straße"", ""reagiert beim A...",None,2,0.72,"[""man lying on the street"", ""reacts when spoke...",None,1,0.62,"[""Mann liegt auf der Straße vor Primark (Bahnh...",None
3,Notruf_4,4,1,0.78,"[""caller speaks/responds throughout the call"",...",None,2,0.72,"[""Caller reports 'krieg schlecht Luft' (diffic...",None,...,"[""Caller reports no pain"", ""No dizziness/circu...",None,1,0.55,"[""caller answers questions and gives location""...",None,1,0.70,"[""kein Trauma/Unfallmechanismus genannt"", ""Auf...",None
4,Notruf_5,5,1,0.66,"[""Patientin ist normal bei Bewusstsein"", ""Kein...",None,1,0.62,"[""caller reports she sometimes pants a little ...",None,...,"[""normal bei Bewusstsein"", ""keine weiteren Sch...",None,1,0.78,"[""Patientin laut Disponent \""normal bei Bewuss...",None,3,0.64,"[""Schwangere in 34. Schwangerschaftswoche"", ""G...",None
5,Notruf_6,6,3,0.60,"[""Patientin wirkt laut Anruferin, als ob sie n...",None,3,0.90,"[""caller reports patient is not breathing (\""D...",None,...,"[""82-jährige Patientin im Bett liegend vorgefu...",None,3,0.86,"[""Patientin im Bett liegend vorgefunden, wirkt...",None,0,0.72,"[""Patientin 82 Jahre alt im Bett liegend aufge...",None
6,Notruf_7,7,3,0.78,"[""offensichtliche Messerstichverletzung am Hal...",None,3,0.90,"[""Person hat \""schlecht Luft bis kaum noch Luf...",None,...,"[""offensichtliche Messerstichverletzung am Hal...",None,3,0.74,"[""Person sitzt noch aufrecht"", ""Sehr eingetrüb...",None,3,0.86,"[""Person mit offensichtlicher Messerstichverle...",None
7,Notruf_8,8,1,0.62,"[""Caller speaks in full sentences"", ""No mouth/...",None,0,0.55,[],None,...,"[""Sturz von Leiter/Dach aus ca. 3 Metern"", ""st...",None,0,0.34,[],None,3,0.73,"[""Sturz von einer Leiter beim Versuch aufs Dac...",None
8,Notruf_9,9,2,0.72,"[""speaking still possible"", ""reports very diff...",None,2,0.74,"[""kriegt ganz schlecht Luft"", ""kann noch sprec...",None,...,[],None,1,0.72,"[""reagiert beim Ansprechen ganz normal"", ""kann...",None,2,0.65,"[""Atemnot nach dem Essen von Nusskuchen"", ""Ges...",None
9,Notruf_10,10,2,0.66,"[""Frau ist beim Bewusstsein"", ""redet undeutlic...",None,0,0.15,[],None,...,"[""Frau ist verwirrt"", ""spricht undeutlich/nusc...",None,2,0.85,"[""seit dem Aufstehen ganz komisch"", ""beim Bewu...",None,0,0.72,[],None
